#Prepare Calendar Table

This notebook build a standard calendar dimension table covering date from 1940-01-01 to 2030-12-31

In [0]:
create schema if not exists adb_healthcare.silver;

In [0]:
create or replace table adb_healthcare.silver.calendar as
with dates as (
    select explode (
        sequence (
        to_date('1940-01-01'),
        to_date('2030-12-31'),
            interval 1 day
        )
    ) as calendar_date
)
select
    cast(date_format(calendar_date, 'yyyyMMdd') as int) as date_key,
    calendar_date as date,
    year(calendar_date) as year,
    quarter(calendar_date) as quarter,
    ((month(calendar_date)-1)%3) + 1 as month_of_quarter,
    month(calendar_date) as month,
    date_format(calendar_date, 'MMMM') as month_name,
    date_format(calendar_date, 'MMM') as month_name_short,
    concat(year(calendar_date), '-', lpad(month(calendar_date),2,'0')) as year_month,
    concat(year(calendar_date), lpad(month(calendar_date),2,'0')) as year_month_key,
    weekofyear(calendar_date) as week_of_year,
    concat(year(calendar_date), '-W', lpad(weekofyear(calendar_date),2,'0')) as year_week,
    day(calendar_date) as day_of_month,
    dayofyear(calendar_date) as day_of_year,
    weekday(calendar_date)+1 as iso_day_of_week,
    dayofweek(calendar_date) as us_day_of_week,
    date_format(calendar_date, 'EEEE') AS day_name,
    date_format(calendar_date, 'E') AS day_name_short,
    CASE WHEN dayofweek(calendar_date) IN (1, 7) THEN true ELSE false END AS is_weekend,
    CASE WHEN calendar_date = trunc(calendar_date, 'month') THEN true ELSE false END AS is_month_start,
    CASE WHEN calendar_date = last_day(calendar_date) THEN true ELSE false END AS is_month_end,
    CASE WHEN calendar_date = trunc(calendar_date, 'quarter') THEN true ELSE false END AS is_quarter_start,
    CASE WHEN calendar_date = date_sub(add_months(trunc(calendar_date, 'quarter'), 3), 1) THEN true ELSE false END AS is_quarter_end,
    CASE WHEN calendar_date = trunc(calendar_date, 'year') THEN true ELSE false END AS is_year_start,
    CASE WHEN calendar_date = date_sub(add_months(trunc(calendar_date, 'year'), 12), 1) THEN true ELSE false END AS is_year_end,
    trunc(calendar_date, 'week') AS week_start_date,
    date_add(trunc(calendar_date, 'week'), 6) AS week_end_date,
    trunc(calendar_date, 'month') AS month_start_date,
    last_day(calendar_date) AS month_end_date,
    trunc(calendar_date, 'quarter') AS quarter_start_date,
    date_sub(add_months(trunc(calendar_date, 'quarter'), 3), 1) AS quarter_end_date,
    trunc(calendar_date, 'year') AS year_start_date,
    date_sub(add_months(trunc(calendar_date, 'year'), 12), 1) AS year_end_date,
    CASE WHEN (year(calendar_date) % 400 = 0) OR (year(calendar_date) % 4 = 0 AND year(calendar_date) % 100 <> 0) THEN true ELSE false END AS is_leap_year
from dates;

In [0]:
SELECT
  COUNT(*) AS total_rows,
  MIN(date) AS min_date,
  MAX(date) AS max_date,
  MIN(date_key) AS min_date_key,
  MAX(date_key) AS max_date_key,
  COUNT(DISTINCT year) AS distinct_years,
  COUNT(DISTINCT year_month_key) AS distinct_months
FROM adb_healthcare.silver.calendar;

In [0]:
select * from adb_healthcare.silver.calendar order by date;